# Section 2.1 Diagnostics — all completed runs

Standalone diagnostics: scans every `completed_*` run folder on Google Drive, loads **every**
checkpoint of **every** run/method, and computes the Section 2.1 metrics (effective rank, within-N
KNN, top-20 singular values). For each checkpoint it writes a traceable
`section21_diagnostics_epoch<E>.csv` + `section21_top20_singular_values_epoch<E>.png` into that run's
own log folder, and a combined `analysis/section21_all_diagnostics.csv` across everything.

Run order: Drive mount -> repo sync -> CIFAR-10 -> diagnostics. No training is performed, and it does
**not** depend on any in-memory `results`, so it works in a fresh session after long runs finish.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is only available inside Google Colab; skipping Drive mount.')


In [ ]:
import getpass
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

PUBLIC_REPO_URL = 'https://github.com/moucheng2017/instance_self_sup.git'
BRANCH = '2-vicreg-barlow-twins-baselines'  # Use 'main' after this baseline suite is merged.
REPO_DIR = Path('/content/instance_self_sup')
GITHUB_TOKEN = ''  # Set to a token string here if the repo is private.


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass

repo_url = PUBLIC_REPO_URL
if GITHUB_TOKEN:
    repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(GITHUB_TOKEN, safe="")}@')


def sync_repo():
    if GITHUB_TOKEN:
        run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_DIR)
    else:
        print('No GitHub token found; keeping the existing origin URL for fetch/pull.')
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

os.chdir('/content')

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
else:
    sync_repo()

os.chdir(REPO_DIR)
run(['python', '-m', 'pip', 'install', '-r', 'requirements_colab.txt'])
print('Repository is ready at', REPO_DIR)


In [ ]:
# CIFAR-10 must be present for feature extraction. Download into the SAME data dir the
# training runs used: /content/drive/MyDrive/<PROJECT_NAME>/data (train_from_colab default).
import os
import torchvision

PROJECT_NAME = 'InstanceSelfSup_CIFAR10'  # must match the training notebooks' PROJECT_NAME
DATA_DIR = f'/content/drive/MyDrive/{PROJECT_NAME}/data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Checking / downloading CIFAR-10 ...')
torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
print('CIFAR-10 is ready at', DATA_DIR)

In [ ]:
import glob
import json
import os
import re

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml

from analysis.spectral import build_diagnostics_loader, extract_features, knn_eval, spectral_diagnostics
from arguments import build_args
from datasets.subset import load_subset_indices, select_subset_indices
from linear_eval import load_backbone_weights
from models import get_backbone

# ---- Drive layout (matches colab_utils.default_colab_paths) ----
DRIVE_ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
LOGS_DIR = os.path.join(DRIVE_ROOT, 'logs')
CKPT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
SUMMARY_DIR = os.path.join(DRIVE_ROOT, 'analysis')
SUBSET_SEED = 42  # the shared subset seed used by every training notebook
os.makedirs(SUMMARY_DIR, exist_ok=True)


def parse_run_dir(log_dir):
    """completed_<ts>_<config_stem>-N<n>-bs<bs>  ->  run dict (or None if it doesn't match)."""
    name = os.path.basename(log_dir.rstrip('/'))
    match = re.match(r'completed_(\d+)_(.+)-N([^-]+)-bs(\d+)$', name)
    if not match:
        return None
    timestamp, config_stem, n_raw, batch_size = match.group(1), match.group(2), match.group(3), int(match.group(4))
    config_file = f'configs/baselines/{config_stem}.yaml'
    if not os.path.exists(config_file):
        return None
    return {
        'timestamp': timestamp,
        'config_stem': config_stem,
        'config_file': config_file,
        'N': n_raw if n_raw == 'full' else int(n_raw),
        'subset_n': None if n_raw == 'full' else int(n_raw),
        'batch_size': batch_size,
        'run_name': f'{config_stem}-N{n_raw}-bs{batch_size}',
        'log_dir': log_dir,
    }


def epoch_from_checkpoint(path):
    match = re.search(r'_epoch(\d+)_', os.path.basename(path))
    return int(match.group(1)) if match else -1


def checkpoints_for_run(run):
    """All checkpoints for a run, sorted by epoch. Prefer <log_dir>/checkpoint_path.txt;
    fall back to globbing the shared checkpoints/ folder by run name (covers resumed runs)."""
    paths = []
    listing = os.path.join(run['log_dir'], 'checkpoint_path.txt')
    if os.path.exists(listing):
        with open(listing) as handle:
            paths = [line.strip() for line in handle if line.strip()]
    if not paths:
        paths = glob.glob(os.path.join(CKPT_DIR, f"{run['run_name']}_epoch*.pth"))
    paths = [p for p in dict.fromkeys(paths) if os.path.exists(p)]
    return sorted(paths, key=epoch_from_checkpoint)


def method_name(config_file):
    with open(config_file) as handle:
        return yaml.safe_load(handle)['model']['name']


def selected_indices_for_run(run, args):
    idx_path = os.path.join(run['log_dir'], 'subset_indices.json')
    if os.path.exists(idx_path):
        return load_subset_indices(idx_path)
    if run['subset_n'] is None:
        return None
    probe = build_diagnostics_loader(args, indices=None, batch_size=run['batch_size'])
    return select_subset_indices(len(probe.dataset), run['subset_n'], SUBSET_SEED)


def save_checkpoint_figure(method, n, batch_size, epoch, singular_values, log_dir):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(1, len(singular_values) + 1), singular_values, marker='o')
    ax.set_xlabel('Singular value rank')
    ax.set_ylabel('Singular value')
    ax.set_title(f'{method}  N={n}  bs={batch_size}  epoch={epoch}')
    ax.grid(True, alpha=0.3)
    png_path = os.path.join(log_dir, f'section21_top20_singular_values_epoch{epoch}.png')
    fig.savefig(png_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return png_path


def diagnose_all(logs_dir):
    all_rows = []
    run_dirs = sorted(glob.glob(os.path.join(logs_dir, 'completed_*')))
    print(f'Found {len(run_dirs)} completed_* folder(s) under {logs_dir}')
    for log_dir in run_dirs:
        run = parse_run_dir(log_dir)
        if run is None:
            print(f'  skip (unrecognized name / no config): {os.path.basename(log_dir)}')
            continue
        method = method_name(run['config_file'])
        checkpoints = checkpoints_for_run(run)
        if not checkpoints:
            print(f"  skip (no checkpoints): {run['run_name']}")
            continue
        overrides = {'train': {'subset_n': run['subset_n'], 'subset_seed': SUBSET_SEED, 'batch_size': run['batch_size']}}
        args = build_args(
            config_file=run['config_file'], overrides=overrides,
            data_dir=DATA_DIR, log_dir='/tmp/diag_logs', ckpt_dir='/tmp/diag_ckpts',
            device='cuda' if torch.cuda.is_available() else 'cpu',
            download=False, create_dirs=False,
        )
        indices = selected_indices_for_run(run, args)
        loader = build_diagnostics_loader(args, indices=indices, batch_size=run['batch_size'])
        n_samples = len(loader.dataset) if run['subset_n'] is None else run['subset_n']
        for ckpt in checkpoints:
            epoch = epoch_from_checkpoint(ckpt)
            backbone = get_backbone(args.model.backbone).to(args.device)
            load_backbone_weights(backbone, ckpt)
            features, labels = extract_features(backbone, loader, args.device, n_samples=n_samples)
            singular_values, erank, _ = spectral_diagnostics(features)
            n_train = int(0.8 * len(features))
            knn = knn_eval(features, labels, k=min(20, n_train), n_train=n_train)
            row = {
                'method': method, 'config': run['config_stem'], 'N': run['N'],
                'batch_size': run['batch_size'], 'epoch': epoch,
                'effective_rank': erank, 'knn_accuracy': knn,
                'run_dir': os.path.basename(log_dir), 'checkpoint': os.path.basename(ckpt),
            }
            all_rows.append(row)
            # Per-checkpoint outputs saved into the run's own log_dir (traceable by epoch).
            pd.DataFrame([row]).to_csv(os.path.join(log_dir, f'section21_diagnostics_epoch{epoch}.csv'), index=False)
            save_checkpoint_figure(method, run['N'], run['batch_size'], epoch, singular_values[:20], log_dir)
            print(f"  {method} N={run['N']} bs={run['batch_size']} epoch={epoch}: erank={erank:.2f} knn={knn:.3f}")
    return pd.DataFrame(all_rows)


diagnostics_table = diagnose_all(LOGS_DIR)
display(diagnostics_table)

# One combined, sortable summary across every method / run / checkpoint.
summary_csv = os.path.join(SUMMARY_DIR, 'section21_all_diagnostics.csv')
diagnostics_table.to_csv(summary_csv, index=False)
print(f'\nSaved combined summary across all runs -> {summary_csv}')